In [1]:
import qiskit
import numpy as np
import matplotlib.pyplot as plt
from itertools import product
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator
from qiskit.visualization import plot_histogram

print("Qiskit Version:", qiskit.__version__)
simulator = AerSimulator()

def apply_constant_oracle(qc, n, value):
    if value == 1:
        qc.x(n)

def apply_parity_oracle(qc, n):
    for q in range(n):
        qc.cx(q, n)

def apply_truth_table_oracle(qc, n, outputs):
    for x, fx in enumerate(outputs):
        if fx == 1:
            bits = format(x, f"0{n}b")
            for q, bit in enumerate(bits):
                if bit == "0": qc.x(q)
            qc.mcx(list(range(n)), n)
            for q, bit in enumerate(bits):
                if bit == "0": qc.x(q)

def deutsch_jozsa(n, oracle_type="constant", constant_value=0, outputs=None, shots=1024):
    qc = QuantumCircuit(n+1, n)
    qc.x(n)
    qc.h(range(n+1))
    if oracle_type == "constant": apply_constant_oracle(qc,n,constant_value)
    elif oracle_type == "parity": apply_parity_oracle(qc,n)
    elif oracle_type == "truth_table": apply_truth_table_oracle(qc,n,outputs)
    else: raise ValueError("Unknown oracle type")
    qc.h(range(n))
    qc.measure(range(n),range(n))
    counts = simulator.run(qc,shots=shots).result().get_counts()
    return qc, counts

def classify_result(counts,n):
    return "Constant" if counts.get("0"*n,0)==sum(counts.values()) else "Balanced"

Qiskit Version: 2.5.1


In [2]:
rng=np.random.default_rng(42)
def random_balanced_function(n):
    total=2**n; outputs=[0]*total
    for i in rng.choice(total,size=total//2,replace=False): outputs[i]=1
    return outputs

def run_random_tests(n=4,trials=5):
    correct=0
    for t in range(1,trials+1):
        outputs=random_balanced_function(n)
        _,counts=deutsch_jozsa(n,"truth_table",outputs=outputs,shots=2048)
        predicted=classify_result(counts,n)
        ok=predicted=="Balanced"
        correct+=ok
        print(f"Trial {t}: ones={sum(outputs)}, classification={predicted}, correct={ok}")
    print("Accuracy:",100*correct/trials,"%.")
run_random_tests()

Trial 1: ones=8, classification=Balanced, correct=True
Trial 2: ones=8, classification=Balanced, correct=True
Trial 3: ones=8, classification=Balanced, correct=True
Trial 4: ones=8, classification=Balanced, correct=True
Trial 5: ones=8, classification=Balanced, correct=True
Accuracy: 100.0 %.
